# Génération des périodes de transfert

Ce notebook transforme un fichier Excel contenant au minimum :

- `id_tiers`
- `date_debut`
- `date_fin`

en une ligne par **mois de transfert**.

### Règles
- **P2** : mois de début du contrat.
- **P1** : mois de fin du contrat.
- **COMPLET** : mois entièrement couvert → taux = 100 %.
- **P2/P1** : taux = nombre de jours couverts / nombre réel de jours du mois.
- Si le contrat commence le 1er jour du mois, le mois est considéré **COMPLET**.
- Si le contrat finit le dernier jour du mois, le mois est considéré **COMPLET**.
- Si début et fin sont dans le même mois, la partie est **P2_P1** si le mois n'est pas complet.
- Les dates `date_debut` et `date_fin` d'origine sont conservées dans chaque ligne.


In [ ]:
import pandas as pd
import calendar
from pathlib import Path

# =========================
# PARAMÈTRES
# =========================
FICHIER_ENTREE = "periode_transfert.xlsx"
FICHIER_SORTIE = "periodes_transfert_detaillees.xlsx"

COL_ID = "id_tiers"
COL_DEBUT = "date_debut"
COL_FIN = "date_fin"


In [ ]:
# =========================
# LECTURE DU FICHIER
# =========================
df = pd.read_excel(FICHIER_ENTREE)

colonnes_requises = [COL_ID, COL_DEBUT, COL_FIN]
manquantes = [c for c in colonnes_requises if c not in df.columns]

if manquantes:
    raise ValueError(
        f"Colonnes manquantes : {manquantes}. "
        f"Colonnes trouvées : {list(df.columns)}"
    )

df[COL_DEBUT] = pd.to_datetime(df[COL_DEBUT], dayfirst=True, errors="coerce")
df[COL_FIN] = pd.to_datetime(df[COL_FIN], dayfirst=True, errors="coerce")

# Contrôles
if df[COL_DEBUT].isna().any() or df[COL_FIN].isna().any():
    erreurs = df[df[COL_DEBUT].isna() | df[COL_FIN].isna()]
    display(erreurs)
    raise ValueError("Certaines dates sont invalides ou vides.")

if (df[COL_FIN] < df[COL_DEBUT]).any():
    erreurs = df[df[COL_FIN] < df[COL_DEBUT]]
    display(erreurs)
    raise ValueError("Certaines date_fin sont antérieures aux date_debut.")

print(f"{len(df):,} contrats chargés.")
display(df.head())


In [ ]:
# =========================
# GÉNÉRATION DES PÉRIODES
# =========================
def generer_periodes(ligne):
    id_tiers = ligne[COL_ID]
    date_debut = ligne[COL_DEBUT].normalize()
    date_fin = ligne[COL_FIN].normalize()

    premier_mois = date_debut.to_period("M")
    dernier_mois = date_fin.to_period("M")

    resultat = []

    for periode in pd.period_range(premier_mois, dernier_mois, freq="M"):
        debut_mois = periode.start_time.normalize()
        fin_mois = periode.end_time.normalize()

        # Intersection entre le contrat et le mois
        debut_eligible = max(date_debut, debut_mois)
        fin_eligible = min(date_fin, fin_mois)

        nb_jours_mois = calendar.monthrange(periode.year, periode.month)[1]
        nb_jours = (fin_eligible - debut_eligible).days + 1
        taux = nb_jours / nb_jours_mois

        est_premier = periode == premier_mois
        est_dernier = periode == dernier_mois
        mois_complet = debut_eligible == debut_mois and fin_eligible == fin_mois

        if mois_complet:
            partie = "COMPLET"
        elif est_premier and est_dernier:
            partie = "P2_P1"
        elif est_premier:
            partie = "P2"
        elif est_dernier:
            partie = "P1"
        else:
            partie = "COMPLET"

        resultat.append({
            "id_tiers": id_tiers,
            "date_debut": date_debut,
            "date_fin": date_fin,
            "periode": debut_mois,
            "mois": periode.month,
            "annee": periode.year,
            "partie": partie,
            "date_debut_periode": debut_eligible,
            "date_fin_periode": fin_eligible,
            "nb_jours": nb_jours,
            "nb_jours_mois": nb_jours_mois,
            "taux": taux
        })

    return resultat


lignes_sortie = []

for _, ligne in df.iterrows():
    lignes_sortie.extend(generer_periodes(ligne))

resultat = pd.DataFrame(lignes_sortie)

resultat = resultat.sort_values(
    ["id_tiers", "date_debut", "periode"]
).reset_index(drop=True)

print(f"{len(resultat):,} périodes générées.")
display(resultat.head(20))


In [ ]:
# =========================
# CONTRÔLES DE COHÉRENCE
# =========================
assert (resultat["nb_jours"] >= 1).all()
assert (resultat["nb_jours"] <= resultat["nb_jours_mois"]).all()
assert ((resultat["taux"] > 0) & (resultat["taux"] <= 1)).all()

# Un mois COMPLET doit avoir un taux de 100 %
assert (
    resultat.loc[resultat["partie"] == "COMPLET", "taux"].round(10) == 1
).all()

print("Contrôles de cohérence : OK")


In [ ]:
# =========================
# EXPORT EXCEL
# =========================
with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    resultat.to_excel(writer, sheet_name="Periodes", index=False)

    ws = writer.book["Periodes"]

    # Formats dates
    for col in ["B", "C", "D", "H", "I"]:
        for cell in ws[col][1:]:
            cell.number_format = "dd/mm/yyyy"

    # Taux en pourcentage
    for cell in ws["L"][1:]:
        cell.number_format = "0.00%"

    # Filtre et gel de la première ligne
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"

    # Largeurs simples
    largeurs = {
        "A": 18, "B": 14, "C": 14, "D": 14,
        "E": 10, "F": 10, "G": 14, "H": 20,
        "I": 20, "J": 12, "K": 15, "L": 12
    }
    for col, largeur in largeurs.items():
        ws.column_dimensions[col].width = largeur

print(f"Fichier créé : {FICHIER_SORTIE}")
